In [ ]:
# 1. Install required packages
!pip install -q textblob vaderSentiment
!pip install -q transformers

# 2. Imports
import pandas as pd
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt2
import seaborn as sns
import re

# 3. VADER
vader_analyzer = SentimentIntensityAnalyzer()

# 4. User token
from google.colab import userdata
userdata.get('...') # ENTER THE NAME OF YOUR SECRET KEY HERE (A HUGGING FACE USER TOKEN!)

# 5. Load data
from google.colab import files
uploaded = files.upload()
df = pd.read_csv(next(iter(uploaded)))

# 6. Rows with dialogues, mental state and personality
dialogues = df[(df["event"] == "dialogue") & df["mental_state"].notna() & df["personality"].notna()].copy()
print(f"{len(dialogues)} párbeszédes esemény betöltve.")

# 7. Split text
def extract_agent_text(content, which='first'):
    try:
        match = re.match(r"(Agent_\w+)\s+talks with\s+(Agent_\w+):", content)
        if not match:
            return None
        agent1, agent2 = match.groups()
        quoted_lines = re.findall(r'"(.*?)"', content)
        if not quoted_lines:
            return None
        selected_agent = agent1 if which == 'first' else agent2
        relevant_lines = [line for line in quoted_lines if line.strip().startswith(f"{selected_agent}:")]
        return " ".join(relevant_lines).replace(f"{selected_agent}:", "").strip()
    except:
        return None

dialogues["first_agent_text"] = dialogues["content"].apply(lambda x: extract_agent_text(x, which="first"))
dialogues["second_agent_text"] = dialogues["content"].apply(lambda x: extract_agent_text(x, which="second"))

# 8. TextBlob sentiment
def textblob_sentiment(text):
    try:
        blob = TextBlob(text)
        return blob.sentiment.polarity, blob.sentiment.subjectivity
    except:
        return None, None

dialogues[["tb_polarity", "tb_subjectivity"]] = (
    dialogues["first_agent_text"].apply(textblob_sentiment).apply(pd.Series)
)

# 9. BERT sentiment
bert_pipe = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")

def get_bert_sentiment(text):
    try:
        result = bert_pipe(text[:512])[0]
        return result['label'], result['score']
    except:
        return None, None

dialogues[["bert_label", "bert_score"]] = dialogues["first_agent_text"].apply(get_bert_sentiment).apply(pd.Series)

# 10. VADER sentiment (compound)
def vader_sentiment(text):
    try:
        vs = vader_analyzer.polarity_scores(text)
        return vs["compound"]
    except:
        return None

dialogues["vader_compound"] = dialogues["first_agent_text"].apply(vader_sentiment)

# 11. Sample of results
display(dialogues[["first_agent_text", "tb_polarity", "bert_label", "vader_compound", "mental_state", "personality"]].sample(5))

# 12. Aggregated matrix: tb_polarity és vader_compound
agg_mean = dialogues.groupby(["mental_state", "personality"])[["tb_polarity", "vader_compound", "bert_score"]].mean()
agg_std = dialogues.groupby(["mental_state", "personality"])[["tb_polarity", "vader_compound", "bert_score"]].std()

# Format: "M ± SD", rounded to 3 decimal points
agg = agg_mean.round(3).astype(str) + " ± " + agg_std.round(3).astype(str)

display(agg.sort_values("tb_polarity", ascending=False))

agg = dialogues.groupby(["mental_state", "personality"])[["tb_polarity", "vader_compound", "bert_score"]].mean().round(3)
display(agg.sort_values("tb_polarity", ascending=False))